# Trees, Binary Search Trees, Heaps and Tries

Notebook for **Chapter 7** of the *dsa* study document. Run the cells top to bottom. Read the full explanation on the site: https://dadipavan.github.io/Learning/study/dsa/ch07.html

Tip: change a number, predict the output, then run the cell again.

## 7.1 Binary Tree Anatomy

In [ ]:
class Node:
    def __init__(self, val, left=None, right=None):
        self.val, self.left, self.right = val, left, right

def tree(t):
    # Build a tree from nested tuples: int = leaf, (val, left, right), None = empty
    if t is None:
        return None
    if isinstance(t, int):
        return Node(t)
    return Node(t[0], tree(t[1]), tree(t[2]))

def count(n):
    # How many nodes? Empty tree has 0; otherwise 1 + both sides.
    return 0 if n is None else 1 + count(n.left) + count(n.right)

def height(n):
    # Height in EDGES. An empty tree gets -1 so a single node gets 0.
    if n is None:
        return -1
    return 1 + max(height(n.left), height(n.right))

root = tree((8, (3, 1, (6, 4, 7)), (10, None, (14, 13, None))))
print("root value:", root.val)
print("left child of root:", root.left.val)
print("right child of root:", root.right.val)
print("left child of 10:", root.right.left)
print("nodes:", count(root), "height:", height(root))
assert count(root) == 9 and height(root) == 3

root value: 8
left child of root: 3
right child of root: 10
left child of 10: None
nodes: 9 height: 3


In [ ]:
import math

print("perfect tree: height -> nodes")
for h in range(5):
    print(h, "->", 2 ** (h + 1) - 1)

print("n nodes -> smallest possible height")
for n in [1, 7, 1000, 1000000]:
    print(n, "->", math.ceil(math.log2(n + 1)) - 1)
print("skewed tree with 1000000 nodes has height", 1000000 - 1)
assert math.ceil(math.log2(1000 + 1)) - 1 == 9

perfect tree: height -> nodes
0 -> 1
1 -> 3
2 -> 7
3 -> 15
4 -> 31
n nodes -> smallest possible height
1 -> 0
7 -> 2
1000 -> 9
1000000 -> 19
skewed tree with 1000000 nodes has height 999999


In [ ]:
# This cell is meant to fail: read the error message, that is the lesson.
class Node:
    def __init__(self, val, left=None, right=None):
        self.val, self.left, self.right = val, left, right

def height(n):
    if n is None:
        return -1
    return 1 + max(height(n.left), height(n.right))

# A "tree" that is really a chain going left, 5000 nodes long.
head = None
for v in range(5000):
    head = Node(v, left=head)

try:
    height(head)
except RecursionError:
    print("RecursionError: too deep for plain recursion")

RecursionError: too deep for plain recursion


## 7.2 Depth-First Search: Preorder, Inorder, Postorder

In [ ]:
class Node:
    def __init__(self, val, left=None, right=None):
        self.val, self.left, self.right = val, left, right

def tree(t):
    # Build a tree from nested tuples: int = leaf, (val, left, right), None = empty
    if t is None:
        return None
    if isinstance(t, int):
        return Node(t)
    return Node(t[0], tree(t[1]), tree(t[2]))

def preorder(n, out):
    if n is None:            # base case: nothing to visit
        return
    out.append(n.val)        # VISIT first
    preorder(n.left, out)
    preorder(n.right, out)

def inorder(n, out):
    if n is None:
        return
    inorder(n.left, out)
    out.append(n.val)        # VISIT in the middle
    inorder(n.right, out)

def postorder(n, out):
    if n is None:
        return
    postorder(n.left, out)
    postorder(n.right, out)
    out.append(n.val)        # VISIT last

root = tree((8, (3, 1, (6, 4, 7)), (10, None, (14, 13, None))))
for f in (preorder, inorder, postorder):
    out = []
    f(root, out)
    print(f.__name__.ljust(10), out)
assert inorder.__name__ == "inorder"

preorder   [8, 3, 1, 6, 4, 7, 10, 14, 13]
inorder    [1, 3, 4, 6, 7, 8, 10, 13, 14]
postorder  [1, 4, 7, 6, 3, 13, 14, 10, 8]


In [ ]:
class Node:
    def __init__(self, val, left=None, right=None):
        self.val, self.left, self.right = val, left, right

def tree(t):
    # Build a tree from nested tuples: int = leaf, (val, left, right), None = empty
    if t is None:
        return None
    if isinstance(t, int):
        return Node(t)
    return Node(t[0], tree(t[1]), tree(t[2]))

def preorder_iter(root):
    out, stack = [], [root] if root else []
    while stack:
        n = stack.pop()
        out.append(n.val)                 # visit when popped
        if n.right: stack.append(n.right) # right goes in first...
        if n.left:  stack.append(n.left)  # ...so left comes out first
    return out

def inorder_iter(root):
    out, stack, cur = [], [], root
    while cur or stack:
        while cur:                        # slide down the left edge
            stack.append(cur)
            cur = cur.left
        cur = stack.pop()                 # leftmost unvisited node
        out.append(cur.val)               # visit it
        cur = cur.right                   # then deal with its right side
    return out

def postorder_iter(root):
    # Trick: "node, right, left" reversed is "left, right, node".
    out, stack = [], [root] if root else []
    while stack:
        n = stack.pop()
        out.append(n.val)
        if n.left:  stack.append(n.left)
        if n.right: stack.append(n.right)
    return out[::-1]

root = tree((8, (3, 1, (6, 4, 7)), (10, None, (14, 13, None))))
print(preorder_iter(root))
print(inorder_iter(root))
print(postorder_iter(root))
assert preorder_iter(root) == [8, 3, 1, 6, 4, 7, 10, 14, 13]
assert inorder_iter(root) == [1, 3, 4, 6, 7, 8, 10, 13, 14]
assert postorder_iter(root) == [1, 4, 7, 6, 3, 13, 14, 10, 8]

[8, 3, 1, 6, 4, 7, 10, 14, 13]
[1, 3, 4, 6, 7, 8, 10, 13, 14]
[1, 4, 7, 6, 3, 13, 14, 10, 8]


In [ ]:
def solve(node):
    if node is None:
        return BASE_VALUE               # answer for an empty tree
    left = solve(node.left)             # trust recursion: answer for left subtree
    right = solve(node.right)           # answer for right subtree
    return COMBINE(node.val, left, right)

In [ ]:
class Node:
    def __init__(self, val, left=None, right=None):
        self.val, self.left, self.right = val, left, right

def tree(t):
    # Build a tree from nested tuples: int = leaf, (val, left, right), None = empty
    if t is None:
        return None
    if isinstance(t, int):
        return Node(t)
    return Node(t[0], tree(t[1]), tree(t[2]))

def max_depth(n):
    if n is None:
        return 0
    return 1 + max(max_depth(n.left), max_depth(n.right))

print(max_depth(tree((8, (3, 1, (6, 4, 7)), (10, None, (14, 13, None))))))
print(max_depth(None), max_depth(tree(5)), max_depth(tree((1, (2, (3, 4, None), None), None))))
assert max_depth(None) == 0 and max_depth(tree(5)) == 1

4
0 1 4


## 7.3 Breadth-First Search: Level Order

In [ ]:
class Node:
    def __init__(self, val, left=None, right=None):
        self.val, self.left, self.right = val, left, right

def tree(t):
    # Build a tree from nested tuples: int = leaf, (val, left, right), None = empty
    if t is None:
        return None
    if isinstance(t, int):
        return Node(t)
    return Node(t[0], tree(t[1]), tree(t[2]))
from collections import deque

def level_order(root):
    if root is None:
        return []
    result, queue = [], deque([root])
    while queue:
        size = len(queue)              # nodes on THIS level, frozen now
        level = []
        for _ in range(size):
            node = queue.popleft()     # O(1), unlike list.pop(0)
            level.append(node.val)
            if node.left:
                queue.append(node.left)    # next level
            if node.right:
                queue.append(node.right)
        result.append(level)
    return result

root = tree((8, (3, 1, (6, 4, 7)), (10, None, (14, 13, None))))
print(level_order(root))
assert level_order(root) == [[8], [3, 10], [1, 6, 14], [4, 7, 13]]
assert level_order(None) == []

[[8], [3, 10], [1, 6, 14], [4, 7, 13]]


In [ ]:
class Node:
    def __init__(self, val, left=None, right=None):
        self.val, self.left, self.right = val, left, right

def tree(t):
    # Build a tree from nested tuples: int = leaf, (val, left, right), None = empty
    if t is None:
        return None
    if isinstance(t, int):
        return Node(t)
    return Node(t[0], tree(t[1]), tree(t[2]))
from collections import deque

def right_side_view(root):
    if root is None:
        return []
    view, queue = [], deque([root])
    while queue:
        size = len(queue)
        for i in range(size):
            node = queue.popleft()
            if i == size - 1:              # last node of this level
                view.append(node.val)
            if node.left:
                queue.append(node.left)
            if node.right:
                queue.append(node.right)
    return view

print(right_side_view(tree((8, (3, 1, (6, 4, 7)), (10, None, (14, 13, None))))))
print(right_side_view(tree((1, (2, (3, None, None), None), None))))
assert right_side_view(None) == []

[8, 10, 14, 13]
[1, 2, 3]


## 7.4 Return-Value Recursion: Balanced, Diameter, Path Sum, LCA, Serialize

In [ ]:
def solve(root):
    best = 0                            # side answer, updated as we go
    def dfs(node):
        nonlocal best
        if node is None:
            return BASE
        l, r = dfs(node.left), dfs(node.right)
        best = max(best, CANDIDATE(l, r))   # best answer that turns at this node
        return EXTENDABLE(l, r)             # what the parent is allowed to extend
    dfs(root)
    return best

In [ ]:
class Node:
    def __init__(self, val, left=None, right=None):
        self.val, self.left, self.right = val, left, right

def tree(t):
    # Build a tree from nested tuples: int = leaf, (val, left, right), None = empty
    if t is None:
        return None
    if isinstance(t, int):
        return Node(t)
    return Node(t[0], tree(t[1]), tree(t[2]))

def is_balanced(root):
    FAIL = -2
    def height(n):
        if n is None:
            return -1
        l = height(n.left)
        if l == FAIL:
            return FAIL                # propagate failure up immediately
        r = height(n.right)
        if r == FAIL or abs(l - r) > 1:
            return FAIL
        return 1 + max(l, r)
    return height(root) != FAIL

print(is_balanced(tree((8, (3, 1, (6, 4, 7)), (10, None, (14, 13, None))))))
print(is_balanced(tree((1, (2, 3, 4), 5))))
print(is_balanced(None), is_balanced(tree((1, (2, 3, None), None))))
assert is_balanced(tree((1, (2, 3, 4), 5)))

False
True
True False


In [ ]:
class Node:
    def __init__(self, val, left=None, right=None):
        self.val, self.left, self.right = val, left, right

def tree(t):
    # Build a tree from nested tuples: int = leaf, (val, left, right), None = empty
    if t is None:
        return None
    if isinstance(t, int):
        return Node(t)
    return Node(t[0], tree(t[1]), tree(t[2]))

def diameter(root):
    best = 0
    def height(n):
        nonlocal best
        if n is None:
            return -1
        l, r = height(n.left), height(n.right)
        best = max(best, l + r + 2)       # path that turns at n, counted in edges
        return 1 + max(l, r)              # what the parent may extend
    height(root)
    return best

def diameter_slow(root):
    # brute force for checking: heights recomputed at every node
    def h(n):
        return -1 if n is None else 1 + max(h(n.left), h(n.right))
    if root is None:
        return 0
    here = h(root.left) + h(root.right) + 2
    return max(here, diameter_slow(root.left), diameter_slow(root.right))

a = tree((8, (3, 1, (6, 4, 7)), (10, None, (14, 13, None))))
b = tree((1, (2, (3, 4, 5), (6, 7, 8)), None))     # big part hangs off the left only
print(diameter(a), diameter(b), diameter(tree(9)))
assert diameter(a) == diameter_slow(a) == 6
assert diameter(b) == diameter_slow(b) == 4

6 4 0


In [ ]:
class Node:
    def __init__(self, val, left=None, right=None):
        self.val, self.left, self.right = val, left, right

def tree(t):
    # Build a tree from nested tuples: int = leaf, (val, left, right), None = empty
    if t is None:
        return None
    if isinstance(t, int):
        return Node(t)
    return Node(t[0], tree(t[1]), tree(t[2]))

def lca(root, p, q):
    if root is None or root.val == p or root.val == q:
        return root                        # found one target, or fell off
    left = lca(root.left, p, q)
    right = lca(root.right, p, q)
    if left and right:                     # one target on each side: split point
        return root
    return left or right                   # otherwise pass up whichever side found something

root = tree((8, (3, 1, (6, 4, 7)), (10, None, (14, 13, None))))
for p, q in [(4, 7), (4, 13), (3, 7), (1, 4), (13, 14)]:
    print(p, q, "->", lca(root, p, q).val)
assert lca(root, 4, 7).val == 6 and lca(root, 4, 13).val == 8 and lca(root, 3, 7).val == 3

4 7 -> 6
4 13 -> 8
3 7 -> 3
1 4 -> 3
13 14 -> 14


In [ ]:
class Node:
    def __init__(self, val, left=None, right=None):
        self.val, self.left, self.right = val, left, right

def tree(t):
    # Build a tree from nested tuples: int = leaf, (val, left, right), None = empty
    if t is None:
        return None
    if isinstance(t, int):
        return Node(t)
    return Node(t[0], tree(t[1]), tree(t[2]))

def serialize(root):
    out = []
    def pre(n):
        if n is None:
            out.append("#")
            return
        out.append(str(n.val))
        pre(n.left)
        pre(n.right)
    pre(root)
    return ",".join(out)

def deserialize(s):
    tokens = iter(s.split(","))
    def build():
        tok = next(tokens)               # shared position in the token stream
        if tok == "#":
            return None
        node = Node(int(tok))
        node.left = build()              # left subtree uses the next tokens
        node.right = build()
        return node
    return build()

root = tree((8, (3, 1, (6, 4, 7)), (10, None, (14, 13, None))))
text = serialize(root)
print(text)
again = deserialize(text)
print(serialize(again) == text, serialize(None), serialize(tree((-5, 12, None))))
assert serialize(deserialize(text)) == text

8,3,1,#,#,6,4,#,#,7,#,#,10,#,14,13,#,#,#
True # -5,12,#,#,#


## 7.5 Binary Search Trees

In [ ]:
class Node:
    def __init__(self, val, left=None, right=None):
        self.val, self.left, self.right = val, left, right

def tree(t):
    # Build a tree from nested tuples: int = leaf, (val, left, right), None = empty
    if t is None:
        return None
    if isinstance(t, int):
        return Node(t)
    return Node(t[0], tree(t[1]), tree(t[2]))

def search(root, key):
    cur = root
    while cur:
        if key == cur.val:
            return cur
        cur = cur.left if key < cur.val else cur.right   # ordering throws half away
    return None

def insert(root, key):
    # Returns the (possibly new) root of this subtree. USE the return value.
    if root is None:
        return Node(key)
    if key < root.val:
        root.left = insert(root.left, key)
    elif key > root.val:
        root.right = insert(root.right, key)
    return root                                          # equal keys: ignore

def inorder(n):
    return [] if n is None else inorder(n.left) + [n.val] + inorder(n.right)

root = None
for v in [8, 3, 10, 1, 6, 14, 4, 7, 13]:
    root = insert(root, v)
print(inorder(root))
print(search(root, 7).val, search(root, 5))
assert inorder(root) == sorted([8, 3, 10, 1, 6, 14, 4, 7, 13])

[1, 3, 4, 6, 7, 8, 10, 13, 14]
7 None


In [ ]:
class Node:
    def __init__(self, val, left=None, right=None):
        self.val, self.left, self.right = val, left, right

def tree(t):
    # Build a tree from nested tuples: int = leaf, (val, left, right), None = empty
    if t is None:
        return None
    if isinstance(t, int):
        return Node(t)
    return Node(t[0], tree(t[1]), tree(t[2]))

def is_bst(n, low=float("-inf"), high=float("inf")):
    if n is None:
        return True
    if not (low < n.val < high):                 # violates an ancestor's bound
        return False
    return is_bst(n.left, low, n.val) and is_bst(n.right, n.val, high)

def is_bst_wrong(n):
    # looks only at parent and child: the classic bug
    if n is None:
        return True
    if n.left and n.left.val >= n.val: return False
    if n.right and n.right.val <= n.val: return False
    return is_bst_wrong(n.left) and is_bst_wrong(n.right)

bad = tree((5, 1, (8, 4, 9)))
good = tree((8, (3, 1, (6, 4, 7)), (10, None, (14, 13, None))))
print("good:", is_bst(good), "| bad:", is_bst(bad), "| wrong checker says bad is", is_bst_wrong(bad))
print("duplicate:", is_bst(tree((2, 2, None))), "| empty:", is_bst(None))
assert is_bst(good) and not is_bst(bad) and is_bst_wrong(bad)

good: True | bad: False | wrong checker says bad is True
duplicate: False | empty: True


In [ ]:
class Node:
    def __init__(self, val, left=None, right=None):
        self.val, self.left, self.right = val, left, right

def tree(t):
    # Build a tree from nested tuples: int = leaf, (val, left, right), None = empty
    if t is None:
        return None
    if isinstance(t, int):
        return Node(t)
    return Node(t[0], tree(t[1]), tree(t[2]))

def delete(node, key):
    if node is None:
        return None                            # key not found
    if key < node.val:
        node.left = delete(node.left, key)     # reconnect with the returned subtree
    elif key > node.val:
        node.right = delete(node.right, key)
    else:
        if node.left is None:                  # 0 or 1 child: lift the other side
            return node.right
        if node.right is None:
            return node.left
        succ = node.right                      # 2 children: smallest in right subtree
        while succ.left:
            succ = succ.left
        node.val = succ.val
        node.right = delete(node.right, succ.val)
    return node

def preorder(n):
    return [] if n is None else [n.val] + preorder(n.left) + preorder(n.right)

def inorder(n):
    return [] if n is None else inorder(n.left) + [n.val] + inorder(n.right)

def fresh():
    return tree((8, (3, 1, (6, 4, 7)), (10, None, (14, 13, None))))

for key in [7, 14, 3, 8, 99]:
    t = delete(fresh(), key)
    print("delete", str(key).rjust(2), "-> preorder", preorder(t), "inorder sorted:", inorder(t) == sorted(inorder(t)))
assert preorder(delete(fresh(), 3)) == [8, 4, 1, 6, 7, 10, 14, 13]

delete  7 -> preorder [8, 3, 1, 6, 4, 10, 14, 13] inorder sorted: True
delete 14 -> preorder [8, 3, 1, 6, 4, 7, 10, 13] inorder sorted: True
delete  3 -> preorder [8, 4, 1, 6, 7, 10, 14, 13] inorder sorted: True
delete  8 -> preorder [10, 3, 1, 6, 4, 7, 14, 13] inorder sorted: True
delete 99 -> preorder [8, 3, 1, 6, 4, 7, 10, 14, 13] inorder sorted: True


In [ ]:
class Node:
    def __init__(self, val, left=None, right=None):
        self.val, self.left, self.right = val, left, right

def tree(t):
    # Build a tree from nested tuples: int = leaf, (val, left, right), None = empty
    if t is None:
        return None
    if isinstance(t, int):
        return Node(t)
    return Node(t[0], tree(t[1]), tree(t[2]))

def kth_smallest(root, k):
    stack, cur = [], root
    while cur or stack:
        while cur:
            stack.append(cur)
            cur = cur.left
        cur = stack.pop()
        k -= 1
        if k == 0:                  # this is the k-th node in sorted order
            return cur.val
        cur = cur.right
    return None                     # k was larger than the number of nodes

root = tree((8, (3, 1, (6, 4, 7)), (10, None, (14, 13, None))))
print([kth_smallest(root, k) for k in (1, 3, 5, 9, 10)])
assert [kth_smallest(root, k) for k in range(1, 10)] == [1, 3, 4, 6, 7, 8, 10, 13, 14]

[1, 4, 7, 14, None]


## 7.6 Heaps and Priority Queues

In [ ]:
class MinHeap:
    def __init__(self, items=None):
        self.a = list(items or [])      # assume items already form a valid heap
        self.log = []

    def push(self, x):
        a = self.a
        a.append(x)                      # 1. put at the end
        i = len(a) - 1
        while i > 0:
            p = (i - 1) // 2             # parent index
            if a[p] <= a[i]:
                break
            self.log.append(f"swap index {i} (value {a[i]}) with parent index {p} (value {a[p]})")
            a[i], a[p] = a[p], a[i]      # 2. sift up
            i = p

    def pop(self):
        a = self.a
        top = a[0]
        last = a.pop()                   # remove the last item...
        if a:
            a[0] = last                  # ...and drop it in the root hole
            i, n = 0, len(a)
            while True:                  # sift down
                l, r, m = 2 * i + 1, 2 * i + 2, i
                if l < n and a[l] < a[m]: m = l
                if r < n and a[r] < a[m]: m = r
                if m == i:
                    break
                self.log.append(f"swap index {i} (value {a[i]}) with child index {m} (value {a[m]})")
                a[i], a[m] = a[m], a[i]
                i = m
        return top

h = MinHeap([2, 5, 3, 9, 6, 8, 4])
h.push(1)
print("after push(1):", h.a)
print("popped:", h.pop())
print("after pop:    ", h.a)
print("\n".join(h.log))

import heapq, random
random.seed(7)
data = [random.randint(0, 99) for _ in range(200)]
mine, ref = MinHeap(), []
for x in data:
    mine.push(x)
    heapq.heappush(ref, x)
assert mine.a == ref
assert [mine.pop() for _ in range(200)] == sorted(data)

after push(1): [1, 2, 3, 5, 6, 8, 4, 9]
popped: 1
after pop:     [2, 5, 3, 9, 6, 8, 4]
swap index 7 (value 1) with parent index 3 (value 9)
swap index 3 (value 1) with parent index 1 (value 5)
swap index 1 (value 1) with parent index 0 (value 2)
swap index 0 (value 9) with child index 1 (value 2)
swap index 1 (value 9) with child index 3 (value 5)


In [ ]:
import heapq

h = []
for x in [5, 3, 8, 1, 9]:
    heapq.heappush(h, x)
    print("push", x, "->", h)
print("peek:", h[0], "| list is not sorted, but the first item is the minimum")
out = []
while h:
    out.append(heapq.heappop(h))
print("pops in order:", out)

# Max-heap: store negatives, negate again on the way out.
mx = []
for x in [5, 3, 8, 1, 9]:
    heapq.heappush(mx, -x)
print("largest:", -heapq.heappop(mx), "then", -heapq.heappop(mx))

# Tuples compare field by field: (priority, name)
tasks = []
heapq.heappush(tasks, (2, "write report"))
heapq.heappush(tasks, (1, "fix outage"))
heapq.heappush(tasks, (2, "answer email"))
print([heapq.heappop(tasks) for _ in range(3)])

# heapify: O(n) in-place
arr = [7, 9, 3, 8, 1, 5]
heapq.heapify(arr)
print("heapify:", arr)
assert arr[0] == 1

push 5 -> [5]
push 3 -> [3, 5]
push 8 -> [3, 5, 8]
push 1 -> [1, 3, 8, 5]
push 9 -> [1, 3, 8, 5, 9]
peek: 1 | list is not sorted, but the first item is the minimum
pops in order: [1, 3, 5, 8, 9]
largest: 9 then 8
[(1, 'fix outage'), (2, 'answer email'), (2, 'write report')]
heapify: [1, 7, 3, 8, 9, 5]


In [ ]:
import heapq

def kth_largest(nums, k):
    heap = []                                   # min-heap of the k largest so far
    for x in nums:
        if len(heap) < k:
            heapq.heappush(heap, x)
        elif x > heap[0]:
            heapq.heapreplace(heap, x)          # pop the smallest, push x
    return heap[0]

print(kth_largest([3, 2, 1, 5, 6, 4], 2))
print(kth_largest([5, 5, 3], 2), kth_largest([9], 1), kth_largest([4, 1, 7], 3))
assert kth_largest([3, 2, 1, 5, 6, 4], 2) == sorted([3, 2, 1, 5, 6, 4])[-2]

5
5 9 1


In [ ]:
import heapq
from collections import Counter

def top_k_frequent(words, k):
    counts = Counter(words)
    # smallest key = highest count, then alphabetical
    return heapq.nsmallest(k, counts, key=lambda w: (-counts[w], w))

words = "the day is sunny the the the sunny is is".split()
print(top_k_frequent(words, 2))
print(top_k_frequent(["b", "a", "b", "a", "c"], 2))
print(Counter(words).most_common(2))
assert top_k_frequent(words, 4) == ["the", "is", "sunny", "day"]

['the', 'is']
['a', 'b']
[('the', 4), ('is', 3)]


In [ ]:
import heapq

class ListNode:
    def __init__(self, val=0, next=None):
        self.val, self.next = val, next

def make(values):
    head = None
    for v in reversed(values):
        head = ListNode(v, head)
    return head

def to_list(head):
    out = []
    while head:
        out.append(head.val)
        head = head.next
    return out

def merge_k(lists):
    heap = []
    for i, node in enumerate(lists):
        if node:                                    # skip empty lists
            heapq.heappush(heap, (node.val, i, node))
    dummy = tail = ListNode()
    while heap:
        _, i, node = heapq.heappop(heap)            # smallest current head
        tail.next = node
        tail = node
        if node.next:
            heapq.heappush(heap, (node.next.val, i, node.next))
    return dummy.next

merged = merge_k([make([1, 4, 5]), make([1, 3, 4]), make([2, 6])])
print(to_list(merged))
print(to_list(merge_k([None, make([7]), make([])])), to_list(merge_k([])))
assert to_list(merge_k([make([1, 4, 5]), make([1, 3, 4]), make([2, 6])])) == [1, 1, 2, 3, 4, 4, 5, 6]

[1, 1, 2, 3, 4, 4, 5, 6]
[7] []


In [ ]:
import heapq

class MedianFinder:
    def __init__(self):
        self.low = []     # max-heap (store negatives): the smaller half
        self.high = []    # min-heap: the larger half

    def add(self, x):
        # 1. Always go through the lower half first...
        heapq.heappush(self.low, -x)
        # 2. ...then move its biggest to the upper half, so every low <= every high.
        heapq.heappush(self.high, -heapq.heappop(self.low))
        # 3. Keep sizes balanced: low may have one extra, never fewer.
        if len(self.high) > len(self.low):
            heapq.heappush(self.low, -heapq.heappop(self.high))

    def median(self):
        if len(self.low) > len(self.high):
            return float(-self.low[0])
        return (-self.low[0] + self.high[0]) / 2

mf = MedianFinder()
for x in [5, 2, 8, 1, 9, 3]:
    mf.add(x)
    print(f"add {x}: lower half {sorted(-v for v in mf.low)}, upper half {sorted(mf.high)}, median {mf.median()}")

data = [5, 2, 8, 1, 9, 3]
check = MedianFinder()
for i, x in enumerate(data, 1):
    check.add(x)
    s = sorted(data[:i])
    expected = float(s[i // 2]) if i % 2 else (s[i // 2 - 1] + s[i // 2]) / 2
    assert check.median() == expected

add 5: lower half [5], upper half [], median 5.0
add 2: lower half [2], upper half [5], median 3.5
add 8: lower half [2, 5], upper half [8], median 5.0
add 1: lower half [1, 2], upper half [5, 8], median 3.5
add 9: lower half [1, 2, 5], upper half [8, 9], median 5.0
add 3: lower half [1, 2, 3], upper half [5, 8, 9], median 4.0


## 7.7 Tries: Prefix Trees and Autocomplete

In [ ]:
class TrieNode:
    def __init__(self):
        self.children = {}        # letter -> TrieNode
        self.is_end = False       # does a word finish at this node?

class Trie:
    def __init__(self):
        self.root = TrieNode()

    def insert(self, word):
        node = self.root
        for ch in word:
            if ch not in node.children:            # create the path as needed
                node.children[ch] = TrieNode()
            node = node.children[ch]
        node.is_end = True

    def _walk(self, text):
        node = self.root
        for ch in text:
            node = node.children.get(ch)
            if node is None:
                return None                        # path breaks: nothing has this prefix
        return node

    def search(self, word):                        # exact word?
        node = self._walk(word)
        return node is not None and node.is_end

    def starts_with(self, prefix):                 # any word begins with this?
        return self._walk(prefix) is not None

t = Trie()
for w in ["car", "cat", "cap", "do", "dog"]:
    t.insert(w)
print(t.search("car"), t.search("ca"), t.starts_with("ca"), t.search("do"), t.search("dot"), t.starts_with("x"))
assert t.search("dog") and not t.search("ca") and t.starts_with("ca")

True False True True False False


In [ ]:
class TrieNode:
    def __init__(self):
        self.children = {}
        self.is_end = False

def build(words):
    root = TrieNode()
    for w in words:
        node = root
        for ch in w:
            node = node.children.setdefault(ch, TrieNode())
        node.is_end = True
    return root

def autocomplete(root, prefix, limit=3):
    node = root
    for ch in prefix:
        node = node.children.get(ch)
        if node is None:
            return []
    found = []
    def dfs(n, path):
        if len(found) == limit:
            return
        if n.is_end:
            found.append(prefix + path)          # words come out in alphabetical order
        for ch in sorted(n.children):
            dfs(n.children[ch], path + ch)
    dfs(node, "")
    return found

root = build(["car", "cat", "cap", "do", "dog", "carbon", "cart"])
print(autocomplete(root, "ca"))
print(autocomplete(root, "car"))
print(autocomplete(root, "d"), autocomplete(root, "z"), autocomplete(root, ""))
assert autocomplete(root, "ca") == ["cap", "car", "carbon"]

['cap', 'car', 'carbon']
['car', 'carbon', 'cart']
['do', 'dog'] [] ['cap', 'car', 'carbon']


## 7.8 Segment Trees and Fenwick Trees: Awareness

In [ ]:
import random

class Fenwick:
    def __init__(self, values):
        self.n = len(values)
        self.tree = [0] * (self.n + 1)           # 1-indexed
        self.vals = [0] * (self.n + 1)
        for i, v in enumerate(values, 1):
            self.set(i, v)

    def set(self, i, v):
        delta, self.vals[i] = v - self.vals[i], v
        while i <= self.n:
            self.tree[i] += delta
            i += i & -i                          # move up to the next covering entry

    def prefix(self, i):
        s = 0
        while i > 0:
            s += self.tree[i]
            i -= i & -i                          # drop the lowest bit
        return s

    def range_sum(self, l, r):                   # positions l..r, 1-indexed, inclusive
        return self.prefix(r) - self.prefix(l - 1)

a = [3, 1, 4, 1, 5, 9, 2, 6]
fw = Fenwick(a)
print("prefix(7):", fw.prefix(7), "| sum of positions 3..6:", fw.range_sum(3, 6))
fw.set(3, 14)
a[2] = 14
print("after set position 3 to 14:", fw.range_sum(3, 6), "| total:", fw.prefix(8))

random.seed(3)
for _ in range(500):
    i, v = random.randrange(8), random.randrange(-5, 20)
    fw.set(i + 1, v)
    a[i] = v
    l = random.randrange(8)
    r = random.randrange(l, 8)
    assert fw.range_sum(l + 1, r + 1) == sum(a[l:r + 1])
print("500 random updates and queries agree with the slow method")

prefix(7): 25 | sum of positions 3..6: 19
after set position 3 to 14: 29 | total: 41
500 random updates and queries agree with the slow method


## Chapter recap

## Practice set

Try each problem before you open the answer. They climb from recall to expert. The tree used in "running tree" questions is the one from the start of the chapter. Code in answers was run.

In [ ]:
import heapq
h = []
for x in [4, 1, 7, 3]:
    heapq.heappush(h, -x)
print(-h[0], len(h))
print([-heapq.heappop(h) for _ in range(2)])

In [ ]:
class Node:
    def __init__(self, val, left=None, right=None):
        self.val, self.left, self.right = val, left, right

def tree(t):
    # Build a tree from nested tuples: int = leaf, (val, left, right), None = empty
    if t is None:
        return None
    if isinstance(t, int):
        return Node(t)
    return Node(t[0], tree(t[1]), tree(t[2]))

def f(n):
    if n is None:
        return 0
    return 1 + max(f(n.left), f(n.right))

t = tree((1, (2, 3, None), (4, None, None)))
print(f(t), f(t.left), f(t.right.left))

In [ ]:
class Node:
    def __init__(self, val, left=None, right=None):
        self.val, self.left, self.right = val, left, right

def tree(t):
    # Build a tree from nested tuples: int = leaf, (val, left, right), None = empty
    if t is None:
        return None
    if isinstance(t, int):
        return Node(t)
    return Node(t[0], tree(t[1]), tree(t[2]))

def is_bst_wrong(n):
    if n is None:
        return True
    if n.left and n.left.val >= n.val:
        return False
    if n.right and n.right.val <= n.val:
        return False
    return is_bst_wrong(n.left) and is_bst_wrong(n.right)

fooled = tree((10, (5, None, 15), 20))
print(is_bst_wrong(fooled))

def is_bst(n, low=float("-inf"), high=float("inf")):
    return n is None or (low < n.val < high and is_bst(n.left, low, n.val) and is_bst(n.right, n.val, high))

print(is_bst(fooled))

In [ ]:
class Node:
    def __init__(self, val, left=None, right=None):
        self.val, self.left, self.right = val, left, right

def tree(t):
    # Build a tree from nested tuples: int = leaf, (val, left, right), None = empty
    if t is None:
        return None
    if isinstance(t, int):
        return Node(t)
    return Node(t[0], tree(t[1]), tree(t[2]))

def min_depth_wrong(n):
    if n is None:
        return 0
    return 1 + min(min_depth_wrong(n.left), min_depth_wrong(n.right))

def min_depth(n):
    if n is None:
        return 0
    if n.left is None:
        return 1 + min_depth(n.right)      # only one side exists: must use it
    if n.right is None:
        return 1 + min_depth(n.left)
    return 1 + min(min_depth(n.left), min_depth(n.right))

chain = tree((1, (2, None, None), None))
print(min_depth_wrong(chain), min_depth(chain))

In [ ]:
import heapq

class Item:
    def __init__(self, name):
        self.name = name

h = []
heapq.heappush(h, (1, Item("a")))
try:
    heapq.heappush(h, (1, Item("b")))
except TypeError as e:
    print("TypeError")

h2 = []
for i, name in enumerate(["a", "b"]):
    heapq.heappush(h2, (1, i, Item(name)))      # i is a unique tie-breaker
print(len(h2))

In [ ]:
class Node:
    def __init__(self, val, left=None, right=None):
        self.val, self.left, self.right = val, left, right

def tree(t):
    # Build a tree from nested tuples: int = leaf, (val, left, right), None = empty
    if t is None:
        return None
    if isinstance(t, int):
        return Node(t)
    return Node(t[0], tree(t[1]), tree(t[2]))

def insert(root, key):
    if root is None:
        return Node(key)
    if key < root.val:
        insert(root.left, key)        # result thrown away
    else:
        insert(root.right, key)
    return root

def inorder(n):
    return [] if n is None else inorder(n.left) + [n.val] + inorder(n.right)

root = None
for k in [2, 1, 3]:
    root = insert(root, k)
print(inorder(root))

In [ ]:
class TrieNode:
    def __init__(self):
        self.children = {}
        self.is_end = False

root = TrieNode()
for w in ["an", "and"]:
    node = root
    for ch in w:
        node = node.children.setdefault(ch, TrieNode())
    node.is_end = True

def walk(text):
    node = root
    for ch in text:
        node = node.children.get(ch)
        if node is None:
            return None
    return node

print(walk("an").is_end, walk("a").is_end, walk("a") is not None, walk("andy"))

In [ ]:
class Node:
    def __init__(self, val, left=None, right=None):
        self.val, self.left, self.right = val, left, right

def tree(t):
    # Build a tree from nested tuples: int = leaf, (val, left, right), None = empty
    if t is None:
        return None
    if isinstance(t, int):
        return Node(t)
    return Node(t[0], tree(t[1]), tree(t[2]))

def invert(n):
    if n is None:
        return None
    n.left, n.right = invert(n.right), invert(n.left)
    return n

def has_path_sum(n, remaining):
    if n is None:
        return False
    remaining -= n.val
    if n.left is None and n.right is None:
        return remaining == 0
    return has_path_sum(n.left, remaining) or has_path_sum(n.right, remaining)

def preorder(n):
    return [] if n is None else [n.val] + preorder(n.left) + preorder(n.right)

root = tree((8, (3, 1, (6, 4, 7)), (10, None, (14, 13, None))))
print(has_path_sum(root, 21), has_path_sum(root, 18), has_path_sum(root, 45), has_path_sum(None, 0))
print(preorder(invert(root)))
assert has_path_sum(root, 12) and not has_path_sum(root, 8)

True False True False
[8, 10, 14, 13, 3, 6, 7, 4, 1]


In [ ]:
class Node:
    def __init__(self, val, left=None, right=None):
        self.val, self.left, self.right = val, left, right

def tree(t):
    # Build a tree from nested tuples: int = leaf, (val, left, right), None = empty
    if t is None:
        return None
    if isinstance(t, int):
        return Node(t)
    return Node(t[0], tree(t[1]), tree(t[2]))

def max_path_sum(root):
    best = float("-inf")
    def gain(n):
        nonlocal best
        if n is None:
            return 0
        l = max(gain(n.left), 0)          # ignore a side that only hurts
        r = max(gain(n.right), 0)
        best = max(best, n.val + l + r)   # best path that bends at n
        return n.val + max(l, r)          # best chain the parent can extend
    gain(root)
    return best

print(max_path_sum(tree((-10, 9, (20, 15, 7)))))
print(max_path_sum(tree(-3)), max_path_sum(tree((8, (3, 1, (6, 4, 7)), (10, None, (14, 13, None))))))
assert max_path_sum(tree((1, 2, 3))) == 6

42
-3 61


In [ ]:
class Node:
    def __init__(self, val, left=None, right=None):
        self.val, self.left, self.right = val, left, right

def tree(t):
    # Build a tree from nested tuples: int = leaf, (val, left, right), None = empty
    if t is None:
        return None
    if isinstance(t, int):
        return Node(t)
    return Node(t[0], tree(t[1]), tree(t[2]))

def build(pre, ino):
    pos = {v: i for i, v in enumerate(ino)}      # value -> inorder index
    it = iter(pre)
    def go(lo, hi):                              # inorder slice ino[lo:hi]
        if lo >= hi:
            return None
        v = next(it)                             # preorder: next value is the subtree root
        m = pos[v]
        node = Node(v)
        node.left = go(lo, m)                    # left part of the slice
        node.right = go(m + 1, hi)
        return node
    return go(0, len(ino))

def preorder(n):
    return [] if n is None else [n.val] + preorder(n.left) + preorder(n.right)

pre = [8, 3, 1, 6, 4, 7, 10, 14, 13]
ino = [1, 3, 4, 6, 7, 8, 10, 13, 14]
root = build(pre, ino)
print(preorder(root), root.left.val, root.right.val)
assert preorder(root) == pre

[8, 3, 1, 6, 4, 7, 10, 14, 13] 3 10


In [ ]:
import heapq

def sort_k_sorted(a, k):
    heap = a[:k + 1]
    heapq.heapify(heap)                       # smallest of the first k+1 must be the global smallest
    out = []
    for x in a[k + 1:]:
        out.append(heapq.heappushpop(heap, x))
    while heap:
        out.append(heapq.heappop(heap))
    return out

data = [6, 5, 3, 2, 8, 10, 9]
print(sort_k_sorted(data, 3))
assert sort_k_sorted(data, 3) == sorted(data)

[2, 3, 5, 6, 8, 9, 10]


In [ ]:
class Node:
    def __init__(self):
        self.kids = {}
        self.passing = 0                  # how many words go through this node

class CountTrie:
    def __init__(self):
        self.root = Node()

    def insert(self, word):
        node = self.root
        for ch in word:
            node = node.kids.setdefault(ch, Node())
            node.passing += 1

    def count_prefix(self, p):
        node = self.root
        for ch in p:
            node = node.kids.get(ch)
            if node is None:
                return 0
        return node.passing

t = CountTrie()
for w in ["car", "cat", "cap", "do", "dog"]:
    t.insert(w)
print(t.count_prefix("ca"), t.count_prefix("d"), t.count_prefix("dogs"), t.count_prefix(""))
assert t.count_prefix("ca") == 3

3 2 0 0


## Prompt pack

Copy any prompt into an AI assistant. Each one is tuned for this chapter. Paste your own code or answers where the prompt says so.

---
Done? Do the **Practice set** at the end of the chapter, then clear your Review cards on the site.

https://dadipavan.github.io/Learning/study/dsa/ch07.html